# Study-level CXR Findings Generation — Kaggle runner (image pre-processing variant)

Same methodology as the original `StudyLevel_Pipeline` run, with **one extra step (1c)**: before training, every X-ray the model sees (25k training subset, validation and test) is pre-processed once with **light denoising (3×3 median filter) + CLAHE contrast enhancement**. Processed copies go to `outputs/images_proc/`; the original dataset images are not changed. Everything else (subset, model, training settings, generation, metrics) is identical, so the results can be compared directly with run `subset25k_v1`.

Runs the `StudyLevel_Pipeline` scripts on a Kaggle GPU. **Setup, once per notebook:**

1. **Settings → Accelerator:** `GPU T4 x2` (the pipeline uses one GPU). **Settings → Internet:** On (needed to download BLIP-2 from Hugging Face; requires a phone-verified Kaggle account).
2. **Add Input → Datasets:**
   * the MIMIC-CXR dataset that contains `mimic_cxr_aug_train.csv`, `mimic_cxr_aug_validate.csv` and `official_data_iccv_final/files/...`
   * your uploaded **`StudyLevel_Pipeline_ImgProc`** folder (upload it once as a private Dataset: *Datasets → New Dataset → drag the folder*).
   * optional, saves ~70 min: **Add Input → Notebook Output →** the notebook of the original 25k run (it contains `outputs/processed/` with train/val/test CSVs and `train_subset_25k.csv`, so step 1 is skipped and the *same* 25k studies are used).
3. For long training use **Save Version → Save & Run All (Commit)**. It runs in the background for up to 12 h and keeps `/kaggle/working` as the notebook's output.

**Next session (continuing training):** open the notebook, **Add Input → Notebook Output → this notebook** (make sure it points at the *latest* version, since each session's output builds on the previous one), then *Save & Run All* again. The notebook finds the previous run, skips data preparation, and continues training from `last.pt`. When `STATUS.txt` says `complete`, it generates the test reports and computes the metrics.

In [ ]:
# ============================ SETTINGS ============================
RUN_NAME          = "subset25k_imgproc_v1"  # checkpoint folder name; keep it the same across sessions
EPOCHS            = 5
SUBSET_IMAGES     = 25000       # balanced, quality-filtered training subset (select_subset.py); None = full train.csv
SUBSET_IMAGES_PER_STUDY = 1     # 1 = one frontal image per study (25k studies), 2 = frontal + lateral (~16k studies)
MAX_TRAIN_STUDIES = None        # random cap, only used when SUBSET_IMAGES is None
EXTRA_TRAIN_ARGS  = "--use_lora"  # LoRA adapters in the language model; "" = Q-Former only. Also: "--fusion concat"
RUN_EDA           = True        # make the EDA figures (first session only)
CHECK_IMAGES      = True        # prepare_data checks every image exists (slower, safer)
MAX_TEST_STUDIES  = None        # limit test-set generation (None = all 1,172 test studies)
EVAL_ONLY         = False       # True = skip training and evaluate RUN_NAME's saved model as it is now
                                #        (needs the previous session's output as an input)
EVAL_WEIGHTS      = "best.pt"   # which checkpoint EVAL_ONLY uses: "best.pt" or "last.pt"
# image pre-processing (step 1c): settings live in config.py -> PREPROC_* 
#   denoise = "median" 3x3, CLAHE clip 2.0 with 8x8 tiles, longest side <= 512 px
# ==================================================================

In [ ]:
# Kaggle ships an old torchao (0.10) that makes the latest peft crash when LoRA is applied;
# the pipeline does not use torchao, so remove it (train.py runs in a fresh process afterwards)
!pip uninstall -y -q torchao
!pip install -q rouge-score peft bitsandbytes
import nltk
for pkg in ("wordnet", "omw-1.4"):
    nltk.download(pkg, quiet=True)

In [ ]:
# ---- find the pipeline code among the inputs and copy it to a writable folder ----
import os, sys, glob, re, shutil

def input_dirs(root="/kaggle/input", max_depth=6):
    # breadth-first list of input folders, never walking into the 377k-image tree
    level = [root]
    for _ in range(max_depth + 1):
        nxt = []
        for d in level:
            yield d
            try:
                subs = [e for e in os.scandir(d) if e.is_dir()]
            except OSError:
                continue
            nxt += [e.path for e in subs if e.name != "files" and not re.fullmatch(r"[ps]\d+", e.name)]
        level = nxt

INPUT_DIRS = list(input_dirs())

def find_pipeline_dir():
    for d in INPUT_DIRS:
        if all(os.path.isfile(os.path.join(d, f)) for f in ("train.py", "config.py", "prepare_data.py", "model.py", "preprocess_images.py")):
            return d
    raise FileNotFoundError("StudyLevel_Pipeline_ImgProc code not found - add it with 'Add Input'.")

CODE_SRC = find_pipeline_dir()
CODE_DIR = "/kaggle/working/pipeline"
shutil.copytree(CODE_SRC, CODE_DIR, dirs_exist_ok=True,
                ignore=shutil.ignore_patterns("outputs", "__pycache__", "*.ipynb", "*.bat"))
os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)
os.environ["TQDM_MININTERVAL"] = "30"      # fewer progress-bar lines in the commit log
print("pipeline code:", CODE_SRC, "->", CODE_DIR)

import config
print("train csv :", config.RAW_TRAIN_CSV)
print("test csv  :", config.RAW_VAL_CSV)
print("images    :", config.IMAGE_ROOT)
print("outputs   :", config.OUTPUT_ROOT)
assert config.RAW_TRAIN_CSV and config.RAW_VAL_CSV and config.IMAGE_ROOT, "dataset not found - check inputs"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| compute capability:", torch.cuda.get_device_capability(0))

In [ ]:
# ---- look for a previous session's outputs (added via Add Input -> Notebook Output) ----
PREV_RUN = None
PREV_PROCESSED = None
PREV_IMAGES = None
for d in INPUT_DIRS:
    p = d.replace("\\", "/")
    if PREV_RUN is None and p.endswith("outputs/checkpoints/" + RUN_NAME) and os.path.isfile(os.path.join(d, "last.pt")):
        PREV_RUN = d
    if PREV_PROCESSED is None and all(os.path.isfile(os.path.join(d, f))
                                      for f in ("train.csv", "val.csv", "test.csv", "data_summary.json")):
        PREV_PROCESSED = d        # any input holding prepare_data.py's output (notebook output or uploaded dataset)
    if PREV_IMAGES is None and p.endswith("outputs/images_proc") and os.path.isfile(os.path.join(d, "preprocessing_summary.json")):
        PREV_IMAGES = d           # pre-processed images from an earlier session of THIS notebook

RUN_DIR = os.path.join(config.CHECKPOINT_DIR, RUN_NAME)

def status_of(run_dir):
    p = os.path.join(run_dir, "STATUS.txt")
    return open(p).read().strip() if os.path.isfile(p) else "not started"

print("previous run      :", PREV_RUN, "|", status_of(PREV_RUN) if PREV_RUN else "")
print("previous processed:", PREV_PROCESSED)
print("previous images   :", PREV_IMAGES)
if PREV_PROCESSED is None and not os.path.isfile(os.path.join(config.PROCESSED_DIR, "train.csv")):
    print("\n[!] No processed data found in the inputs -> prepare_data.py will run (~1.5 h).")
    print("    To skip it, stop now and Add Input -> an earlier output of this notebook that contains")
    print("    outputs/processed/ (train.csv, val.csv, test.csv, data_summary.json).")
    print("    Input folders seen:")
    for d in INPUT_DIRS:
        if 2 <= d.count("/") <= 5:
            print("      ", d)

In [ ]:
# ---- step 1: data preparation (skipped when a previous session's CSVs are available) ----
if os.path.isfile(os.path.join(config.PROCESSED_DIR, "train.csv")):
    print("processed data already in", config.PROCESSED_DIR)
elif PREV_PROCESSED:
    shutil.copytree(PREV_PROCESSED, config.PROCESSED_DIR, dirs_exist_ok=True)
    print("copied processed data from", PREV_PROCESSED)
else:
    flag = "" if CHECK_IMAGES else "--no_check_images"
    !python prepare_data.py {flag}
    assert _exit_code == 0, "prepare_data.py failed"
print(open(os.path.join(config.PROCESSED_DIR, "data_summary.json")).read()[:1500])

In [ ]:
# ---- step 1b: balanced 25k-image training subset (select_subset.py) ----
SUBSET_CSV = None
if SUBSET_IMAGES:
    tag = f"train_subset_{SUBSET_IMAGES // 1000}k" if SUBSET_IMAGES % 1000 == 0 else f"train_subset_{SUBSET_IMAGES}"
    tag += "_2img" if SUBSET_IMAGES_PER_STUDY == 2 else ""
    SUBSET_CSV = os.path.join(config.PROCESSED_DIR, tag + ".csv")
    if not os.path.isfile(SUBSET_CSV):
        !python select_subset.py --images {SUBSET_IMAGES} --images_per_study {SUBSET_IMAGES_PER_STUDY}
        assert _exit_code == 0, "select_subset.py failed"
    import json
    summ = json.load(open(SUBSET_CSV.replace(".csv", "_summary.json")))
    print(json.dumps(summ["subset"], indent=2))
    fig = os.path.join(config.FIGURES_DIR, tag + "_prevalence.png")
    if os.path.isfile(fig):
        from IPython.display import Image, display
        display(Image(filename=fig, width=620))
else:
    print("SUBSET_IMAGES is None - training on the full train.csv")

In [ ]:
# ---- step 1c: image pre-processing - light denoising + CLAHE (preprocess_images.py) ----
# Processes the images of the training subset AND of val/test once, with identical settings,
# into config.PROC_IMAGE_ROOT. Already processed files are skipped, so re-running is cheap.
import json
TRAIN_CSV_USED = SUBSET_CSV or os.path.join(config.PROCESSED_DIR, "train.csv")
IMAGES_PER_STUDY = SUBSET_IMAGES_PER_STUDY if SUBSET_CSV else config.MAX_IMAGES_PER_STUDY
summary_file = os.path.join(config.PROC_IMAGE_ROOT, "preprocessing_summary.json")
if PREV_IMAGES and not os.path.isfile(summary_file):
    print("copying pre-processed images from", PREV_IMAGES, "...")
    shutil.copytree(PREV_IMAGES, config.PROC_IMAGE_ROOT, dirs_exist_ok=True)
csvs = " ".join([TRAIN_CSV_USED, os.path.join(config.PROCESSED_DIR, "val.csv"), os.path.join(config.PROCESSED_DIR, "test.csv")])
!python preprocess_images.py --csv {csvs} --max_images {IMAGES_PER_STUDY}
assert _exit_code == 0, "preprocess_images.py failed"
print(json.dumps(json.load(open(summary_file)), indent=2)[:1500])
fig = os.path.join(config.FIGURES_DIR, "preprocessing_examples.png")
if os.path.isfile(fig):
    from IPython.display import Image, display
    display(Image(filename=fig, width=620))


In [ ]:
# ---- step 2: EDA figures (first session only) ----
if RUN_EDA and not PREV_RUN:
    !python eda.py --split train
    from IPython.display import Image, display
    for f in sorted(glob.glob(os.path.join(config.FIGURES_DIR, "train_*.png"))):
        display(Image(filename=f, width=520))

In [ ]:
# ---- step 3: training on the PRE-PROCESSED images (stops cleanly after config.MAX_HOURS, resumes next session) ----
if EVAL_ONLY:
    assert PREV_RUN, f"EVAL_ONLY needs an input containing outputs/checkpoints/{RUN_NAME}/last.pt"
    if not os.path.isdir(RUN_DIR):          # copied so this version's output can still be resumed from
        shutil.copytree(PREV_RUN, RUN_DIR)
    print("EVAL_ONLY: training skipped. Model:", RUN_DIR, "|", status_of(RUN_DIR))
elif PREV_RUN and status_of(PREV_RUN).startswith("complete"):
    if not os.path.isdir(RUN_DIR):
        shutil.copytree(PREV_RUN, RUN_DIR)
    print("training already complete:", status_of(RUN_DIR))
else:
    if PREV_RUN:
        args = f"--resume_from {PREV_RUN} --image_root {config.PROC_IMAGE_ROOT}"
    else:
        args = f"--run_name {RUN_NAME} --epochs {EPOCHS} {EXTRA_TRAIN_ARGS} --image_root {config.PROC_IMAGE_ROOT}"
        if SUBSET_CSV:
            args += f" --train_csv {SUBSET_CSV} --max_images {SUBSET_IMAGES_PER_STUDY}"
        elif MAX_TRAIN_STUDIES:
            args += f" --max_train_studies {MAX_TRAIN_STUDIES}"
    !python train.py {args}
    assert _exit_code == 0, "train.py failed - see the error above"
    print("STATUS:", status_of(RUN_DIR))

In [ ]:
# ---- training curve + latest sample reports ----
import pandas as pd
log = pd.read_csv(os.path.join(RUN_DIR, "train_log.csv"))
display(log.tail(15))
samples = sorted(glob.glob(os.path.join(RUN_DIR, "samples_*.txt")), key=os.path.getmtime)
if samples:
    print(open(samples[-1]).read()[:3000])

In [ ]:
# ---- steps 4 + 5: test-set generation and metrics (when training is complete, or EVAL_ONLY) ----
if EVAL_ONLY or status_of(RUN_DIR).startswith("complete"):
    weights = EVAL_WEIGHTS if EVAL_ONLY else "best.pt"
    sub = "eval_" + weights.replace(".pt", "") if EVAL_ONLY else ""     # keeps interim results separate
    out_dir = os.path.join(config.RESULTS_DIR, RUN_NAME, sub)
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, "model_status.txt"), "w") as f:
        f.write(f"weights={weights} | {status_of(RUN_DIR)}\n")
    pred = os.path.join(out_dir, "predictions_test.csv")
    limit = f"--max_studies {MAX_TEST_STUDIES}" if MAX_TEST_STUDIES else ""
    !python generate.py --run_dir {RUN_DIR} --weights {weights} --out_csv {pred} --image_root {config.PROC_IMAGE_ROOT} {limit}
    assert _exit_code == 0, "generate.py failed"
    !python evaluate.py --predictions {pred}
else:
    print("Training not finished yet:", status_of(RUN_DIR))
    print("-> Save Version now, then next session: Add Input -> this notebook's output, and Run All again.")

In [ ]:
# ---- keep the output small: the copied code folder is not needed in the saved output ----
os.chdir("/kaggle/working")
shutil.rmtree(CODE_DIR, ignore_errors=True)
!du -sh /kaggle/working/outputs/* 2>/dev/null